# Task Statement 4.3: Enforce Structured Output Using Tool Use and JSON Schemas

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 4: Prompt Engineering & Structured Output
- **Domain Weight**: 18% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 6 (Structured Data Extraction Pipeline)

### The Core Goal
To master deterministic structured data extraction by configuring **Tool Use with JSON Schema**, enforcing 100% execution via **forced `tool_choice`**, designing schemas with **explicit nullable fields** to prevent hallucination when facts are absent, and implementing **extensible taxonomies** (`enum` with `"Other"`).

### Why This Matters in Production
In enterprise data pipelines (e.g. processing 50,000 invoices, medical records, or legal filings daily), data must load directly into relational databases or downstream Kafka queues.
If an architect relies on prompt-based JSON instructions:
1. **Markdown Formatting Artifacts**: Claude outputs conversational filler (*"Here is the requested JSON:"*) or wraps output in markdown fences (` ```json ... ``` `), breaking downstream `json.loads()` parsers.
2. **Fabrication on Missing Facts**: If a schema marks `tax_id` as a required non-nullable string, but the physical receipt has no tax ID, Claude feels forced by the schema to invent a plausible-looking fake tax ID.
3. **Rigid Taxonomy Breakage**: If an expense category enum only supports `["Travel", "Meals"]` and an employee submits an invoice for `Software Subscriptions`, validation fails completely.
Configuring Tool Use with JSON Schema guarantees 100% syntactic validity, while nullable fields and extensible enums eliminate data fabrication.

### Exam Importance
Task Statement 4.3 is one of the most frequently tested topics on the exam. Questions test why tool calling is preferred over regex for structured output, how to prevent hallucinations on missing document fields (nullable types), and how to configure extensible categorical fields.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. Why Tool Use Outperforms Prompt-Only JSON Mode
- **Prompt-Only JSON Mode**: The model generates JSON as natural language tokens. If token generation drifts or if max tokens is reached, the JSON is malformed or surrounded by prose.
- **Tool Use Extraction**: The Anthropic API's tool-calling engine constrains token generation at the decoding level to strictly match the provided JSON schema. By setting:
  ```python
  tool_choice = {"type": "tool", "name": "extract_invoice"}
  ```
  Claude **cannot output conversational prose**. Its entire response consists solely of the structured tool argument payload, guaranteeing 100% parser-safe JSON.

### 2. Nullable Fields: The Defense Against Data Fabrication
Consider a contract missing a termination notice window:
- **Flawed Schema**: `{"notice_days": {"type": "integer"}}` (Marked as required).
  - *Model Dilemma*: The document doesn't mention notice days, but the schema requires an integer.
  - *Failure*: Claude hallucinates `30` because 30 days is common in contracts.
- **Production Schema (Nullable Defense)**:
  ```json
  {
    "notice_days": {
      "type": ["integer", "null"],
      "description": "Notice period in days required for termination, or null if unstated in document."
    }
  }
  ```
  - *Result*: Claude outputs `"notice_days": null`. Zero hallucination.

### 3. Extensible Taxonomies (The "Other" Pattern)
Real-world documents contain infinite variations. Closed enums break pipelines when encountering novel categories:
```json
{
  "category": {
    "type": "string",
    "enum": ["Hardware", "Software", "Cloud", "Consulting", "Other"]
  },
  "category_other_detail": {
    "type": ["string", "null"],
    "description": "Specify the category if 'Other' was selected, otherwise null."
  }
}
```
This guarantees that novel vendor items are captured with full semantic fidelity without failing JSON schema validation.

---

## 🔑 Key Technical Terminology Explained

- **Tool-Use Structured Output**: Leveraging Claude's native tool-calling capabilities to emit structured JSON arguments matching a predefined schema rather than relying on natural language text.
- **Forced Tool Choice**: Setting `tool_choice: {"type": "tool", "name": "..."}` to force the model to invoke the extraction schema on turn 1 without emitting conversational text.
- **Explicit Nullable Type**: Defining a JSON Schema property with `type: ["<type>", "null"]` to explicitly permit `null` when source documents lack the data.
- **Extensible Taxonomy**: A schema design pattern pairing a controlled `enum` (including an `"Other"` option) with an auxiliary free-text description field.
- **Decoding Constraint**: An architectural guarantee where the API runtime forces model token generation to conform to JSON syntax rules.

---

## 🏗️ Architecture: Schema-Enforced Extraction Flow

```mermaid
flowchart TD
    DocInput["Unstructured Document<br>(PDF / Email / Invoice)"] --> ClaudeTurn["Claude Extraction Request<br>(claude-sonnet-4-5)"]
    
    ClaudeTurn --> ConfigChoice["tool_choice: {'type': 'tool', 'name': 'extract_invoice'}"]
    
    ConfigChoice --> ForceTool["Claude Generates Tool Call Arguments<br>(Strictly typed to JSON Schema)"]
    
    ForceTool --> SchemaCheck{"JSON Schema Validator"}
    
    SchemaCheck -->|"Missing field / Type error"| ReturnError["Return tool_result with isError: true<br>and specific schema violation details"]
    ReturnError --> ClaudeTurn
    
    SchemaCheck -->|"Valid JSON"| BusinessCheck{"Business Rule Validator<br>(e.g. Line item sum == Total)"}
    
    BusinessCheck -->|"Mismatch detected"| Reprompt["Reprompt with validation feedback"]
    Reprompt --> ClaudeTurn
    
    BusinessCheck -->|"All rules pass"| DatabaseCommit["Commit Validated Record to Database"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Extraction Technique | Correct Implementation | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Structured Output Format** | Use **Tool Use with JSON Schema** and forced `tool_choice`. | **TRAP**: Asking for JSON in prompt and using regular expressions to strip markdown code blocks. | Fragile; prompt-only JSON frequently leaks leading conversational text or truncated braces. |
| **Missing Document Facts** | Define fields with **explicit nullable types** (`type: ["string", "null"]`). | **TRAP**: Marking missing fields as required non-nullable strings. | Forces Claude to invent/hallucinate plausible values to satisfy the rigid schema requirements. |
| **Categorical Taxonomies** | Use an **extensible enum pattern** (`["CatA", "CatB", "Other"]` + `other_detail`). | **TRAP**: Using a rigid, closed enum with no fallback options. | Breaks the extraction pipeline whenever a document contains a novel or unexpected category. |

---

## 💡 Architectural Decision Matrix: Extraction Strategies

| Extraction Requirement | Optimal Implementation | Reason |
|---|---|---|
| 100% syntactically valid JSON | Tool Use + `tool_choice` | Eliminates markdown wrapping and syntax parsing errors. |
| Optional or missing document data | `type: ["string", "null"]` | Prevents the model from fabricating false information. |
| High-variety categorization | Enum with "Other" + detail | Retains structured classification while capturing novel categories. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Extraction Reasoning**: Extended Thinking allows Claude to cross-reference multiple pages of a document, reconcile confusing layout tables, and calculate subtotal checks before emitting the extraction tool arguments.
- **Sampling Rules**: Omit `temperature` whenever thinking is enabled.
- **Thinking Blocks**: The assistant's thinking blocks document how each fact was located in the source text before emitting the JSON payload.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_4_3_structured_output_tool_use.ipynb`](ts_4_3_structured_output_tool_use.ipynb):
1. **Step 1: Setup & Tool Definition**: Builds a production invoice extraction tool schema with nullable fields and extensible enums.
2. **Step 2: Forced Tool Execution**: Demonstrates `tool_choice: {"type": "tool", "name": "extract_invoice"}` eliminating conversational filler.
3. **Step 3: Handling Missing Facts**: Shows that missing fields cleanly extract as `null` without fabrication.
4. **Step 4: Extensible Taxonomy**: Demonstrates extracting a novel category into `"Other"` with detailed descriptions.
5. **Step 5: Verification Assertions**: Verifies 100% schema compliance and zero hallucinated values.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 4.3 Structured Output via Tool Calling.")

### 📐 Step 2: Extraction Schema & Programmatic Validator
We define the extraction schema and the validation logic enforcing arithmetic consistency and category constraints.

In [ ]:
# Step 2: Schema & Validation Logic
EXTRACTION_TOOL = {
    "name": "extract_invoice",
    "description": "Extracts structured invoice fields and line items from document text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "purchase_category": {"type": "string", "enum": ["hardware", "software_subscription", "other"]},
            "other_category_detail": {"type": ["string", "null"]},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"},
                    },
                    "required": ["description", "quantity", "unit_price", "total"],
                },
            },
            "stated_total": {"type": "number"},
        },
        "required": ["invoice_number", "vendor_name", "purchase_category", "line_items", "stated_total"],
    },
}

def validate_extraction(data: Dict[str, Any]) -> Tuple[bool, List[str]]:
    """Validates mathematical consistency and conditional field rules."""
    errors = []
    
    # 1. Arithmetic validation
    items = data.get("line_items", [])
    calc_sum = sum(item.get("total", 0.0) for item in items)
    stated = data.get("stated_total", 0.0)
    if abs(calc_sum - stated) > 0.01:
        errors.append(f"Arithmetic Mismatch: Sum of line items ({calc_sum:.2f}) does not match stated invoice total ({stated:.2f}).")

    # 2. Conditional category validation
    cat = data.get("purchase_category")
    if cat == "other" and not data.get("other_category_detail"):
        errors.append("Validation Error: 'other_category_detail' must be provided when purchase_category is 'other'.")

    return len(errors) == 0, errors

print(" Extraction schema and programmatic validator ready.")

### 🧪 Step 3: Interactive Simulation
We test:
1. Mismatched arithmetic -> Caught by validator.
2. Missing conditional category detail -> Caught by validator.
3. Correct payload -> Passes validation.

In [ ]:
# Step 3: Simulation
# 1. Invalid payload (math mismatch)
invalid_payload = {
    "invoice_number": "INV-001",
    "vendor_name": "Acme Corp",
    "purchase_category": "hardware",
    "line_items": [{"description": "Monitor", "quantity": 1, "unit_price": 200.0, "total": 200.0}],
    "stated_total": 350.0,  # Mismatch!
}
ok1, errors1 = validate_extraction(invalid_payload)
print("Math Mismatch Valid:", ok1)
print("Errors Detected:", errors1)

# 2. Valid payload
valid_payload = {
    "invoice_number": "INV-002",
    "vendor_name": "Cloud Services",
    "purchase_category": "software_subscription",
    "line_items": [{"description": "API Subscription", "quantity": 1, "unit_price": 100.0, "total": 100.0}],
    "stated_total": 100.0,
}
ok2, errors2 = validate_extraction(valid_payload)
print("\nValid Payload Result:", ok2)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Validator catches arithmetic discrepancy.
2. Validator catches missing `other_category_detail`.
3. Valid invoice passes without errors.

In [ ]:
# Step 4: Verification Checks
assert ok1 is False
assert any("does not match stated invoice total" in e for e in errors1)

assert ok2 is True
assert len(errors2) == 0

# Check conditional other category
bad_cat = {"purchase_category": "other", "other_category_detail": None, "line_items": [], "stated_total": 0.0}
ok_cat, err_cat = validate_extraction(bad_cat)
assert ok_cat is False
assert any("other_category_detail" in e for e in err_cat)

print(" All verification assertions passed! Structured output validation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use **tool calling schemas** for guaranteed JSON structure instead of parsing markdown JSON blocks.
- **Rule 2**: Programmatically validate extracted math (sum of line items vs stated total).
- **Rule 3**: Feed specific programmatic validation errors back into Claude for targeted self-correction.